In [1]:
txt = """
I chose the per-class TabGAN/DAE splits to reflect the evaluation metrics and per-class fidelity you provided:

TabGAN outperforms DAE on Macro F1, ROC-AUC, reconstruction error and test loss, so it receives the larger share overall and especially for

Backdoor where the TabGAN class-wise confusion matrix and class statistics best match the original data. I allocate slightly more DAE for the none class (45%) because DAE reproduces some benign feature-statistics and idle/charging state relationships very close to the original (helpful to keep benign variability),

whereas for Backdoor TabGAN preserves the original correlations and discriminative patterns better so it receives the larger share (65%).

For syn-flood I use a 60/40 split favoring TabGAN because both generators approximate syn-flood means and variances well, but TabGAN shows better overall predictive performance and lower reconstruction error on that class, while keeping 40% DAE adds alternative modes and reduces the risk of generator-specific artifacts dominating the node dataset.

| Attack type | TabGAN | DAE |
| ----------- | ------ | --- |
| none        | 55%    | 45% |
| Backdoor    | 65%    | 35% |
| syn-flood   | 60%    | 40% |
"""
print(txt)


I chose the per-class TabGAN/DAE splits to reflect the evaluation metrics and per-class fidelity you provided:

TabGAN outperforms DAE on Macro F1, ROC-AUC, reconstruction error and test loss, so it receives the larger share overall and especially for

Backdoor where the TabGAN class-wise confusion matrix and class statistics best match the original data. I allocate slightly more DAE for the none class (45%) because DAE reproduces some benign feature-statistics and idle/charging state relationships very close to the original (helpful to keep benign variability),

whereas for Backdoor TabGAN preserves the original correlations and discriminative patterns better so it receives the larger share (65%).

For syn-flood I use a 60/40 split favoring TabGAN because both generators approximate syn-flood means and variances well, but TabGAN shows better overall predictive performance and lower reconstruction error on that class, while keeping 40% DAE adds alternative modes and reduces the risk o

In [1]:
"""
Synthetic Tabular Data Generation Pipeline (TabGAN + DAE + Original)
-------------------------------------------------------------------
- Reconstructs one-hot encoding and per-class StandardScaler for DAE.
- Decodes through trained TabularDAE and inverts the scaling back to physical units.
- Includes model provenance tracking ('data_source' column).
- Generates datasets for multiple nodes (A–H) with node-specific block sequences.
- Original data files: synthetic_extension/<attack>_part_<N>.csv, where N=1→A, 2→B, ...
"""

import os
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# =============================================================================
# NODE CONFIGURATION (A–H)
# =============================================================================

# Map node letters to part indices (A=1, B=2, ..., H=8)
NODE_TO_PART = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6,
    "G": 7,
    "H": 8,
}

# Define block sequences per node (each list should have 26 entries)
BLOCK_SEQUENCES = {
    "A": [
        "none",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "Backdoor",
        "none",
        "none",
        "Backdoor",
        "Backdoor",
        "Backdoor",
        "Backdoor",
        "syn-flood",
        "none",
        "Backdoor",
    ],

    "B": [
        "none",
        "none",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "Backdoor",
        "none",
        "Backdoor",
        "Backdoor",
        "Backdoor",
        "none",
        "syn-flood",
        "none",
        "Backdoor",
    ],

    "C": [
        "none",
        "none",
        "none",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "none",
        "none",
        "none",
        "Backdoor",
        "none",
        "none",
        "Backdoor",
        "none",
        "none",
        "none",
        "Backdoor",
        "Backdoor",
        "none",
        "none",
        "syn-flood",
        "none",
        "Backdoor",
    ],

    "D": [
        "none",
        "none",
        "none",
        "none",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "none",
        "Backdoor",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "Backdoor",
        "none",
        "Backdoor",
        "none",
        "none",
        "none",
        "syn-flood",
        "none",
        "Backdoor",
    ],

    "E": [
        "none",
        "none",
        "none",
        "none",
        "none",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "none",
        "none",
        "none",
        "none",
        "Backdoor",
        "none",
        "none",
        "none",
        "none",
        "none",
        "syn-flood",
        "none",
        "none",
        "none",
        "syn-flood",
        "none",
        "Backdoor",
    ],

    "F": [
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "Backdoor",
        "none",
        "syn-flood",
        "syn-flood",
        "none",
        "none",
        "syn-flood",
        "none",
        "Backdoor",
    ],

    "G": [
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "syn-flood",
        "syn-flood",
        "none",
        "none",
        "none",
        "none",
        "Backdoor",
        "Backdoor",
        "none",
        "none",
        "none",
        "none",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "none",
        "syn-flood",
        "none",
        "Backdoor",
    ],

    "H": [
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "none",
        "syn-flood",
        "none",
        "none",
        "none",
        "none",
        "none",
        "Backdoor",
        "none",
        "none",
        "Backdoor",
        "none",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "syn-flood",
        "none",
        "Backdoor",
    ],
}

# Select which nodes to generate in this run
NODES_TO_GENERATE = ["A", "B", "C", "D", "E", "F", "G", "H"]

# =============================================================================
# 1. PATH CONFIGURATION (base paths; node-specific paths set inside function)
# =============================================================================

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "dataset_synthetic extension" / "hyperparameter_tuning"
MODELS_DIR = DATA_DIR / "models"

# Base directory for original data (files are directly in this folder)
ORIGINAL_DATA_BASE = Path("synthetic_extension")

MODEL_COLUMNS = [
    "shunt_voltage", "bus_voltage_V", "current_mA",
    "power_mW", "State", "Attack"
]
NUMERIC_COLUMNS = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]

BLOCK_SIZE = 1689

SPLIT_RATIOS = {
    "none": (0.55, 0.45),
    "Backdoor": (0.65, 0.35),
    "syn-flood": (0.60, 0.40),
}

# =============================================================================
# 2. DAE MODEL CLASS (Exact Match to Training)
# =============================================================================

class TabularDAE(nn.Module):
    def __init__(
        self,
        input_dim=6,
        hidden_dims=(32, 16),
        latent_dim=18,
        dropout=0.05,
        **kwargs
    ):
        super(TabularDAE, self).__init__()
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.latent_dim = latent_dim
        self.dropout_rate = dropout

        # Encoder: 6 -> 32 -> 16 -> 18
        self.encoder = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[0], self.hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[1], self.latent_dim)
        )

        # Decoder: 18 -> 16 -> 32 -> 6
        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[1], self.hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[0], self.input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        return self.decoder(latent)

# =============================================================================
# 3. HELPER FUNCTIONS
# =============================================================================

def load_tabgan_model(model_path: Path):
    if not model_path.exists():
        raise FileNotFoundError(f"TabGAN model not found at: {model_path}")
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    print(f"✓ Loaded TabGAN model: {model_path}")
    return model


def load_dae_model(model_path: Path) -> TabularDAE:
    if not model_path.exists():
        raise FileNotFoundError(f"DAE model not found at: {model_path}")
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    
    config = checkpoint.get("model_config", {})
    model = TabularDAE(
        input_dim=config.get("input_dim", 6),
        hidden_dims=config.get("hidden_dims", (32, 16)),
        latent_dim=config.get("latent_dim", 18),
        dropout=config.get("dropout", 0.05)
    )
    
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print(f"✓ Loaded DAE checkpoint: {model_path}")
    return model


def load_original_data(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"Original data file not found at: {file_path}")
    df = pd.read_csv(file_path).drop(columns=["time"], errors="ignore")
    df = df[MODEL_COLUMNS]
    print(f"✓ Loaded original data: {file_path} ({len(df)} rows)")
    return df


def prepare_class_preprocessors(original_data_dict):
    """
    Fits StandardScaler on each attack class matching training preprocessing.
    """
    scalers = {}
    feature_cols_map = {}

    for attack_class, df in original_data_dict.items():
        # One-Hot Encode 'State'
        df_encoded = pd.get_dummies(df, columns=["State"], drop_first=False)
        
        # Ensure all possible state columns exist
        for state_col in ["State_charging", "State_idle"]:
            if state_col not in df_encoded.columns:
                df_encoded[state_col] = 0

        # Maintain exact feature order: 4 numeric + 2 one-hot
        feature_cols = [c for c in df_encoded.columns if c != "Attack"]
        # Sort or align feature columns to match standard order
        state_cols = sorted([c for c in feature_cols if c.startswith("State_")])
        feature_cols = NUMERIC_COLUMNS + state_cols

        scaler = StandardScaler()
        scaler.fit(df_encoded[feature_cols])

        scalers[attack_class] = scaler
        feature_cols_map[attack_class] = feature_cols

    return scalers, feature_cols_map


def calculate_row_counts(block_size: int, tabgan_ratio: float, dae_ratio: float):
    tabgan_rows = round(block_size * tabgan_ratio)
    dae_rows = block_size - tabgan_rows
    return tabgan_rows, dae_rows


def sample_from_tabgan(model, n_samples: int, block_type: str, original_data: pd.DataFrame) -> pd.DataFrame:
    try:
        generator = model.get_object_generator()
        data = original_data[MODEL_COLUMNS].copy()
        categorical_cols = data.select_dtypes(include=["object"]).columns.tolist()

        encoders = {}
        data_encoded = data.copy()

        for col in categorical_cols:
            unique_vals = data_encoded[col].unique()
            encoder = {val: idx for idx, val in enumerate(unique_vals)}
            decoder = {idx: val for idx, val in enumerate(unique_vals)}
            encoders[col] = {"encoder": encoder, "decoder": decoder}
            data_encoded[col] = data_encoded[col].map(encoder)

        target_col = "Attack"
        # NOTE: train_df / test_df are not defined in this snippet;
        # you likely already have them in your real code. Adjust as needed.
        train_df = data_encoded
        test_df = data_encoded

        features, targets = generator.generate_data(
            train_df=train_df.drop(columns=[target_col]),
            target=train_df[[target_col]].reset_index(drop=True),
            test_df=test_df.drop(columns=[target_col]),
            only_generated_data=True,
        )

        if isinstance(features, pd.DataFrame):
            df = features.copy()
            if isinstance(targets, pd.DataFrame):
                df[target_col] = targets[target_col].values
            else:
                df[target_col] = targets
        else:
            df = pd.DataFrame(features)

        for col in categorical_cols:
            if col in df.columns and col in encoders:
                decoder = encoders[col]["decoder"]
                df[col] = df[col].round().astype(int).map(decoder)

        df = df[MODEL_COLUMNS]

        if len(df) > n_samples:
            df = df.sample(n=n_samples, replace=False).reset_index(drop=True)
        elif len(df) < n_samples and len(df) > 0:
            extra = df.sample(n=n_samples - len(df), replace=True)
            df = pd.concat([df, extra], ignore_index=True)
        elif len(df) == 0:
            df = data.sample(n=n_samples, replace=True).reset_index(drop=True)

        df["data_source"] = "TabGAN"
        return df

    except Exception as e:
        print(f"  ✗ Error sampling from TabGAN ({block_type}): {e}")
        raise


def sample_from_dae(
    dae_model: TabularDAE, 
    scaler: StandardScaler, 
    feature_cols: list, 
    n_samples: int, 
    block_type: str, 
    original_data: pd.DataFrame
) -> pd.DataFrame:
    """
    Encodes real scaled samples, injects noise in latent space, decodes,
    and applies scaler.inverse_transform to restore original physical units.
    """
    try:
        # 1. Sample original data and encode
        raw_samples = original_data.sample(n=n_samples, replace=True).reset_index(drop=True)
        df_encoded = pd.get_dummies(raw_samples, columns=["State"], drop_first=False)
        
        for col in feature_cols:
            if col not in df_encoded.columns:
                df_encoded[col] = 0

        x_scaled = scaler.transform(df_encoded[feature_cols])
        x_tensor = torch.tensor(x_scaled, dtype=torch.float32)

        # 2. Latent generation via real manifold + subtle perturbation
        with torch.no_grad():
            latent = dae_model.encoder(x_tensor)
            # Add small noise in latent space (std = 0.05)
            latent_perturbed = latent + torch.randn_like(latent) * 0.05
            reconstructed_scaled = dae_model.decoder(latent_perturbed).detach().cpu().numpy()

        # 3. Inverse transform scaled features back to real physical scale
        reconstructed_physical = scaler.inverse_transform(reconstructed_scaled)
        df_reconstructed = pd.DataFrame(reconstructed_physical, columns=feature_cols)

        # 4. Clamp continuous numeric features to strictly non-negative
        for col in NUMERIC_COLUMNS:
            df_reconstructed[col] = np.clip(df_reconstructed[col], a_min=0, a_max=None)

        # 5. Decode 'State' from one-hot columns (argmax)
        state_cols = [c for c in feature_cols if c.startswith("State_")]
        if state_cols:
            state_idx = np.argmax(df_reconstructed[state_cols].values, axis=1)
            df_reconstructed["State"] = [state_cols[i].replace("State_", "") for i in state_idx]
        else:
            df_reconstructed["State"] = raw_samples["State"].values

        df_reconstructed["Attack"] = block_type
        df_reconstructed["data_source"] = "DAE"

        return df_reconstructed[MODEL_COLUMNS + ["data_source"]]

    except Exception as e:
        print(f"  ✗ Error sampling from DAE ({block_type}): {e}")
        raise


def replace_with_original(block_df: pd.DataFrame, original_slice: pd.DataFrame, attack_type: str) -> pd.DataFrame:
    block = block_df.copy()
    n_orig = min(len(original_slice), len(block))
    if n_orig == 0:
        return block

    replace_idx = np.random.choice(block.index, size=n_orig, replace=False)
    orig_rows = original_slice[MODEL_COLUMNS].iloc[:n_orig].copy()
    orig_rows["data_source"] = "Original"

    block.loc[replace_idx, MODEL_COLUMNS + ["data_source"]] = orig_rows.values
    return block

# =============================================================================
# 4. MAIN PIPELINE (per node)
# =============================================================================

def generate_synthetic_dataset(node_id: str) -> pd.DataFrame:
    """
    Generate final synthetic dataset for a specific node (A–H).
    Original data files: synthetic_extension/<attack>_part_<N>.csv
    where N = NODE_TO_PART[node_id] (A=1, B=2, ...).
    """
    if node_id not in BLOCK_SEQUENCES:
        raise ValueError(f"Unknown node_id: {node_id}")

    BLOCK_SEQUENCE = BLOCK_SEQUENCES[node_id]
    part_index = NODE_TO_PART[node_id]

    # Original data files for this node (part N)
    ORIGINAL_DATA_FILES = {
        "none": ORIGINAL_DATA_BASE / f"none_part_{part_index}.csv",
        "syn-flood": ORIGINAL_DATA_BASE / f"syn-flood_part_{part_index}.csv",
        "Backdoor": ORIGINAL_DATA_BASE / f"Backdoor_part_{part_index}.csv",
    }

    #OUTPUT_FILE = f"Node_{node_id}_final_synthetic_dataset_with_source.csv"
    OUTPUT_DIR = Path("dataset")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    OUTPUT_FILE = OUTPUT_DIR / f"Node_{node_id}_final_synthetic_dataset_with_source.csv"

    TABGAN_MODELS = {
        "none": MODELS_DIR / "tabgan_generator_none.pkl",
        "syn-flood": MODELS_DIR / "tabgan_generator_syn-flood.pkl",
        "Backdoor": MODELS_DIR / "tabgan_generator_Backdoor.pkl",
    }

    DAE_MODELS = {
        "none": MODELS_DIR / "dae_none.pt",
        "syn-flood": MODELS_DIR / "dae_syn-flood.pt",
        "Backdoor": MODELS_DIR / "dae_Backdoor.pt",
    }

    # === Everything below is your existing pipeline logic ===
    print("=" * 80)
    print(f"SYNTHETIC DATA GENERATION PIPELINE (Node {node_id})")
    print("=" * 80)

    print("\nSTEP 1: Loading models")
    print("-" * 80)
    tabgan_models = {atk: load_tabgan_model(p) for atk, p in TABGAN_MODELS.items()}
    dae_models = {atk: load_dae_model(p) for atk, p in DAE_MODELS.items()}

    print("\nSTEP 2: Loading original data & fitting Scalers")
    print("-" * 80)
    original_data = {atk: load_original_data(p) for atk, p in ORIGINAL_DATA_FILES.items()}
    scalers, feature_cols_map = prepare_class_preprocessors(original_data)
    print("✓ Per-class Scalers and Encoders prepared.")

    attack_block_counts = {atk: BLOCK_SEQUENCE.count(atk) for atk in ["none", "syn-flood", "Backdoor"]}
    attack_block_seen = {atk: 0 for atk in ["none", "syn-flood", "Backdoor"]}
    original_cursors = {atk: 0 for atk in ["none", "syn-flood", "Backdoor"]}

    print("\nSTEP 3: Generating blocks")
    print("-" * 80)
    blocks = []
    cols_with_source = MODEL_COLUMNS + ["data_source"]

    for block_index, block_type in enumerate(tqdm(BLOCK_SEQUENCE, desc=f"Generating blocks (Node {node_id})")):
        attack_block_seen[block_type] += 1
        tab_rows, dae_rows = calculate_row_counts(BLOCK_SIZE, *SPLIT_RATIOS[block_type])

        tab_df = sample_from_tabgan(tabgan_models[block_type], tab_rows, block_type, original_data[block_type])
        dae_df = sample_from_dae(
            dae_models[block_type],
            scalers[block_type],
            feature_cols_map[block_type],
            dae_rows,
            block_type,
            original_data[block_type]
        )

        block_df = pd.concat([tab_df[cols_with_source], dae_df[cols_with_source]], ignore_index=True)

        if len(block_df) > BLOCK_SIZE:
            block_df = block_df.iloc[:BLOCK_SIZE].reset_index(drop=True)
        elif len(block_df) < BLOCK_SIZE:
            extra = block_df.sample(n=BLOCK_SIZE - len(block_df), replace=True)
            block_df = pd.concat([block_df, extra], ignore_index=True)

        total_rows = len(original_data[block_type])
        n_blocks = attack_block_counts[block_type]
        rows_per_block = total_rows // n_blocks

        start_idx = original_cursors[block_type]
        end_idx = total_rows if attack_block_seen[block_type] == n_blocks else min(start_idx + rows_per_block, total_rows)

        orig_slice = original_data[block_type].iloc[start_idx:end_idx].reset_index(drop=True)
        original_cursors[block_type] = end_idx

        block_df = replace_with_original(block_df, orig_slice, block_type)
        block_df = block_df.sample(frac=1.0).reset_index(drop=True)
        blocks.append(block_df)

    final_df = pd.concat(blocks, ignore_index=True)

    print("\n" + "=" * 80)
    print(f"NEGATIVE VALUE CHECK REPORT (Node {node_id})")
    print("=" * 80)
    for col in NUMERIC_COLUMNS:
        neg_count = (final_df[col] < 0).sum()
        print(f"Feature '{col}': {neg_count} negative values")

    print(f"\nFinal dataset sample (Head, Node {node_id}):")
    print(final_df.head(10))

    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✓ Saved dataset with provenance column to: {OUTPUT_FILE}")
    return final_df

# =============================================================================
# 5. RUN FOR ALL SELECTED NODES
# =============================================================================

if __name__ == "__main__":
    generated_datasets = {}
    for node_id in NODES_TO_GENERATE:
        print("\n" + "=" * 80)
        print(f"GENERATING NODE {node_id}")
        print("=" * 80)
        df_node = generate_synthetic_dataset(node_id)
        generated_datasets[node_id] = df_node


GENERATING NODE A
SYNTHETIC DATA GENERATION PIPELINE (Node A)

STEP 1: Loading models
--------------------------------------------------------------------------------
✓ Loaded TabGAN model: ../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_none.pkl
✓ Loaded TabGAN model: ../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_syn-flood.pkl
✓ Loaded TabGAN model: ../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_Backdoor.pkl
✓ Loaded DAE checkpoint: ../data/dataset_synthetic extension/hyperparameter_tuning/models/dae_none.pt
✓ Loaded DAE checkpoint: ../data/dataset_synthetic extension/hyperparameter_tuning/models/dae_syn-flood.pt
✓ Loaded DAE checkpoint: ../data/dataset_synthetic extension/hyperparameter_tuning/models/dae_Backdoor.pt

STEP 2: Loading original data & fitting Scalers
--------------------------------------------------------------------------------
✓ Loaded original data: synthetic_ex

Generating blocks (Node A):   0%|                       | 0/26 [00:00<?, ?it/s]

Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/100 [00:00<?, ?it/s]

Generating blocks (Node A):   4%|▌              | 1/26 [00:08<03:38,  8.76s/it]

...

Generating blocks (Node H):  96%|█████████████▍| 25/26 [03:33<00:08,  8.69s/it]

Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/100 [00:00<?, ?it/s]

Generating blocks (Node H): 100%|██████████████| 26/26 [03:47<00:00,  8.76s/it]



NEGATIVE VALUE CHECK REPORT (Node H)
Feature 'shunt_voltage': 0 negative values
Feature 'bus_voltage_V': 0 negative values
Feature 'current_mA': 0 negative values
Feature 'power_mW': 0 negative values

Final dataset sample (Head, Node H):
   shunt_voltage  bus_voltage_V  current_mA     power_mW     State Attack  \
0     428.000000       5.199560  498.000000  2247.000000      idle   none   
1     508.000000       5.200154  495.000000  2528.000000      idle   none   
2     545.000000       5.197000  508.000000  2660.000000      idle   none   
3     470.552399       5.200846  447.645996  2397.888916  charging   none   
4     428.000000       5.206205  460.000000  2587.000000      idle   none   
5     468.860352       5.202398  462.955536  2389.675293  charging   none   
6     441.000000       5.205356  535.000000  2621.000000      idle   none   
7     484.000000       5.204087  446.000000  2617.000000      idle   none   
8     436.000000       5.199275  510.000000  2348.000000      idle 

In [19]:
"""
Synthetic Tabular Data Generation Pipeline (TabGAN + DAE + Original)
-------------------------------------------------------------------
- Reconstructs one-hot encoding and per-class StandardScaler for DAE.
- Decodes through trained TabularDAE and inverts the scaling back to physical units.
- Includes model provenance tracking ('data_source' column).
"""

import os
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# =============================================================================
# 1. PATH CONFIGURATION
# =============================================================================

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "dataset_synthetic extension" / "hyperparameter_tuning"
MODELS_DIR = DATA_DIR / "models"
ORIGINAL_DATA_DIR = Path("synthetic_extension")

TABGAN_MODELS = {
    "none": MODELS_DIR / "tabgan_generator_none.pkl",
    "syn-flood": MODELS_DIR / "tabgan_generator_syn-flood.pkl",
    "Backdoor": MODELS_DIR / "tabgan_generator_Backdoor.pkl",
}

DAE_MODELS = {
    "none": MODELS_DIR / "dae_none.pt",
    "syn-flood": MODELS_DIR / "dae_syn-flood.pt",
    "Backdoor": MODELS_DIR / "dae_Backdoor.pt",
}

ORIGINAL_DATA_FILES = {
    "none": ORIGINAL_DATA_DIR / "none_part_1.csv",
    "syn-flood": ORIGINAL_DATA_DIR / "syn-flood_part_1.csv",
    "Backdoor": ORIGINAL_DATA_DIR / "Backdoor_part_1.csv",
}

OUTPUT_FILE = "A_Node_final_synthetic_dataset_with_source.csv"

# =============================================================================
# 2. GENERATION PARAMETERS
# =============================================================================

BLOCK_SIZE = 1689

BLOCK_SEQUENCE = [
    "none",
    "syn-flood", "syn-flood", "syn-flood", "syn-flood", "syn-flood", "syn-flood", "syn-flood", "syn-flood",
    "none", "none", "none", "none", "none", "none", "none",
    "Backdoor",
    "none", "none",
    "Backdoor", "Backdoor", "Backdoor", "Backdoor",
    "syn-flood",
    "none",
    "Backdoor",
]

SPLIT_RATIOS = {
    "none": (0.55, 0.45),
    "Backdoor": (0.65, 0.35),
    "syn-flood": (0.60, 0.40),
}

MODEL_COLUMNS = [
    "shunt_voltage", "bus_voltage_V", "current_mA",
    "power_mW", "State", "Attack"
]
NUMERIC_COLUMNS = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]

# =============================================================================
# 3. DAE MODEL CLASS (Exact Match to Training)
# =============================================================================

class TabularDAE(nn.Module):
    def __init__(
        self,
        input_dim=6,
        hidden_dims=(32, 16),
        latent_dim=18,
        dropout=0.05,
        **kwargs
    ):
        super(TabularDAE, self).__init__()
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.latent_dim = latent_dim
        self.dropout_rate = dropout

        # Encoder: 6 -> 32 -> 16 -> 18
        self.encoder = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[0], self.hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[1], self.latent_dim)
        )

        # Decoder: 18 -> 16 -> 32 -> 6
        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[1], self.hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[0], self.input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        return self.decoder(latent)

# =============================================================================
# 4. HELPER FUNCTIONS
# =============================================================================

def load_tabgan_model(model_path: Path):
    if not model_path.exists():
        raise FileNotFoundError(f"TabGAN model not found at: {model_path}")
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    print(f"✓ Loaded TabGAN model: {model_path}")
    return model


def load_dae_model(model_path: Path) -> TabularDAE:
    if not model_path.exists():
        raise FileNotFoundError(f"DAE model not found at: {model_path}")
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    
    config = checkpoint.get("model_config", {})
    model = TabularDAE(
        input_dim=config.get("input_dim", 6),
        hidden_dims=config.get("hidden_dims", (32, 16)),
        latent_dim=config.get("latent_dim", 18),
        dropout=config.get("dropout", 0.05)
    )
    
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print(f"✓ Loaded DAE checkpoint: {model_path}")
    return model


def load_original_data(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"Original data file not found at: {file_path}")
    df = pd.read_csv(file_path).drop(columns=["time"], errors="ignore")
    df = df[MODEL_COLUMNS]
    print(f"✓ Loaded original data: {file_path} ({len(df)} rows)")
    return df


def prepare_class_preprocessors(original_data_dict):
    """
    Fits StandardScaler on each attack class matching training preprocessing.
    """
    scalers = {}
    feature_cols_map = {}

    for attack_class, df in original_data_dict.items():
        # One-Hot Encode 'State'
        df_encoded = pd.get_dummies(df, columns=["State"], drop_first=False)
        
        # Ensure all possible state columns exist
        for state_col in ["State_charging", "State_idle"]:
            if state_col not in df_encoded.columns:
                df_encoded[state_col] = 0

        # Maintain exact feature order: 4 numeric + 2 one-hot
        feature_cols = [c for c in df_encoded.columns if c != "Attack"]
        # Sort or align feature columns to match standard order
        state_cols = sorted([c for c in feature_cols if c.startswith("State_")])
        feature_cols = NUMERIC_COLUMNS + state_cols

        scaler = StandardScaler()
        scaler.fit(df_encoded[feature_cols])

        scalers[attack_class] = scaler
        feature_cols_map[attack_class] = feature_cols

    return scalers, feature_cols_map


def calculate_row_counts(block_size: int, tabgan_ratio: float, dae_ratio: float):
    tabgan_rows = round(block_size * tabgan_ratio)
    dae_rows = block_size - tabgan_rows
    return tabgan_rows, dae_rows


def sample_from_tabgan(model, n_samples: int, block_type: str, original_data: pd.DataFrame) -> pd.DataFrame:
    try:
        generator = model.get_object_generator()
        data = original_data[MODEL_COLUMNS].copy()
        categorical_cols = data.select_dtypes(include=["object"]).columns.tolist()

        encoders = {}
        data_encoded = data.copy()

        for col in categorical_cols:
            unique_vals = data_encoded[col].unique()
            encoder = {val: idx for idx, val in enumerate(unique_vals)}
            decoder = {idx: val for idx, val in enumerate(unique_vals)}
            encoders[col] = {"encoder": encoder, "decoder": decoder}
            data_encoded[col] = data_encoded[col].map(encoder)

        train_size = int(len(data_encoded) * 0.8)
        train_df = data_encoded.iloc[:train_size].reset_index(drop=True)
        test_df = data_encoded.iloc[train_size:].reset_index(drop=True)

        target_col = "Attack"
        features, targets = generator.generate_data(
            train_df=train_df.drop(columns=[target_col]),
            target=train_df[[target_col]].reset_index(drop=True),
            test_df=test_df.drop(columns=[target_col]),
            only_generated_data=True,
        )

        if isinstance(features, pd.DataFrame):
            df = features.copy()
            if isinstance(targets, pd.DataFrame):
                df[target_col] = targets[target_col].values
            else:
                df[target_col] = targets
        else:
            df = pd.DataFrame(features)

        for col in categorical_cols:
            if col in df.columns and col in encoders:
                decoder = encoders[col]["decoder"]
                df[col] = df[col].round().astype(int).map(decoder)

        df = df[MODEL_COLUMNS]

        if len(df) > n_samples:
            df = df.sample(n=n_samples, replace=False).reset_index(drop=True)
        elif len(df) < n_samples and len(df) > 0:
            extra = df.sample(n=n_samples - len(df), replace=True)
            df = pd.concat([df, extra], ignore_index=True)
        elif len(df) == 0:
            df = data.sample(n=n_samples, replace=True).reset_index(drop=True)

        df["data_source"] = "TabGAN"
        return df

    except Exception as e:
        print(f"  ✗ Error sampling from TabGAN ({block_type}): {e}")
        raise


def sample_from_dae(
    dae_model: TabularDAE, 
    scaler: StandardScaler, 
    feature_cols: list, 
    n_samples: int, 
    block_type: str, 
    original_data: pd.DataFrame
) -> pd.DataFrame:
    """
    Encodes real scaled samples, injects noise in latent space, decodes,
    and applies scaler.inverse_transform to restore original physical units.
    """
    try:
        # 1. Sample original data and encode
        raw_samples = original_data.sample(n=n_samples, replace=True).reset_index(drop=True)
        df_encoded = pd.get_dummies(raw_samples, columns=["State"], drop_first=False)
        
        for col in feature_cols:
            if col not in df_encoded.columns:
                df_encoded[col] = 0

        x_scaled = scaler.transform(df_encoded[feature_cols])
        x_tensor = torch.tensor(x_scaled, dtype=torch.float32)

        # 2. Latent generation via real manifold + subtle perturbation
        with torch.no_grad():
            latent = dae_model.encoder(x_tensor)
            # Add small noise in latent space (std = 0.05)
            latent_perturbed = latent + torch.randn_like(latent) * 0.05
            reconstructed_scaled = dae_model.decoder(latent_perturbed).detach().cpu().numpy()

        # 3. Inverse transform scaled features back to real physical scale
        reconstructed_physical = scaler.inverse_transform(reconstructed_scaled)
        df_reconstructed = pd.DataFrame(reconstructed_physical, columns=feature_cols)

        # 4. Clamp continuous numeric features to strictly non-negative
        for col in NUMERIC_COLUMNS:
            df_reconstructed[col] = np.clip(df_reconstructed[col], a_min=0, a_max=None)

        # 5. Decode 'State' from one-hot columns (argmax)
        state_cols = [c for c in feature_cols if c.startswith("State_")]
        if state_cols:
            state_idx = np.argmax(df_reconstructed[state_cols].values, axis=1)
            df_reconstructed["State"] = [state_cols[i].replace("State_", "") for i in state_idx]
        else:
            df_reconstructed["State"] = raw_samples["State"].values

        df_reconstructed["Attack"] = block_type
        df_reconstructed["data_source"] = "DAE"

        return df_reconstructed[MODEL_COLUMNS + ["data_source"]]

    except Exception as e:
        print(f"  ✗ Error sampling from DAE ({block_type}): {e}")
        raise


def replace_with_original(block_df: pd.DataFrame, original_slice: pd.DataFrame, attack_type: str) -> pd.DataFrame:
    block = block_df.copy()
    n_orig = min(len(original_slice), len(block))
    if n_orig == 0:
        return block

    replace_idx = np.random.choice(block.index, size=n_orig, replace=False)
    orig_rows = original_slice[MODEL_COLUMNS].iloc[:n_orig].copy()
    orig_rows["data_source"] = "Original"

    block.loc[replace_idx, MODEL_COLUMNS + ["data_source"]] = orig_rows.values
    return block

# =============================================================================
# 5. MAIN PIPELINE
# =============================================================================

def generate_synthetic_dataset():
    print("=" * 80)
    print("SYNTHETIC DATA GENERATION PIPELINE (TabGAN + DAE + Original)")
    print("=" * 80)

    print("\nSTEP 1: Loading models")
    print("-" * 80)
    tabgan_models = {atk: load_tabgan_model(p) for atk, p in TABGAN_MODELS.items()}
    dae_models = {atk: load_dae_model(p) for atk, p in DAE_MODELS.items()}

    print("\nSTEP 2: Loading original data & fitting Scalers")
    print("-" * 80)
    original_data = {atk: load_original_data(p) for atk, p in ORIGINAL_DATA_FILES.items()}
    scalers, feature_cols_map = prepare_class_preprocessors(original_data)
    print("✓ Per-class Scalers and Encoders prepared.")

    attack_block_counts = {atk: BLOCK_SEQUENCE.count(atk) for atk in ["none", "syn-flood", "Backdoor"]}
    attack_block_seen = {atk: 0 for atk in ["none", "syn-flood", "Backdoor"]}
    original_cursors = {atk: 0 for atk in ["none", "syn-flood", "Backdoor"]}

    print("\nSTEP 3: Generating blocks")
    print("-" * 80)
    blocks = []
    cols_with_source = MODEL_COLUMNS + ["data_source"]

    for block_index, block_type in enumerate(tqdm(BLOCK_SEQUENCE, desc="Generating blocks")):
        attack_block_seen[block_type] += 1
        tab_rows, dae_rows = calculate_row_counts(BLOCK_SIZE, *SPLIT_RATIOS[block_type])

        tab_df = sample_from_tabgan(tabgan_models[block_type], tab_rows, block_type, original_data[block_type])
        dae_df = sample_from_dae(
            dae_models[block_type],
            scalers[block_type],
            feature_cols_map[block_type],
            dae_rows,
            block_type,
            original_data[block_type]
        )

        block_df = pd.concat([tab_df[cols_with_source], dae_df[cols_with_source]], ignore_index=True)

        if len(block_df) > BLOCK_SIZE:
            block_df = block_df.iloc[:BLOCK_SIZE].reset_index(drop=True)
        elif len(block_df) < BLOCK_SIZE:
            extra = block_df.sample(n=BLOCK_SIZE - len(block_df), replace=True)
            block_df = pd.concat([block_df, extra], ignore_index=True)

        total_rows = len(original_data[block_type])
        n_blocks = attack_block_counts[block_type]
        rows_per_block = total_rows // n_blocks

        start_idx = original_cursors[block_type]
        end_idx = total_rows if attack_block_seen[block_type] == n_blocks else min(start_idx + rows_per_block, total_rows)

        orig_slice = original_data[block_type].iloc[start_idx:end_idx].reset_index(drop=True)
        original_cursors[block_type] = end_idx

        block_df = replace_with_original(block_df, orig_slice, block_type)
        block_df = block_df.sample(frac=1.0).reset_index(drop=True)
        blocks.append(block_df)

    final_df = pd.concat(blocks, ignore_index=True)

    print("\n" + "=" * 80)
    print("NEGATIVE VALUE CHECK REPORT")
    print("=" * 80)
    for col in NUMERIC_COLUMNS:
        neg_count = (final_df[col] < 0).sum()
        print(f"Feature '{col}': {neg_count} negative values")

    print(f"\nFinal dataset sample (Head):")
    print(final_df.head(10))

    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✓ Saved dataset with provenance column to: {OUTPUT_FILE}")
    return final_df

if __name__ == "__main__":
    generate_synthetic_dataset()

SYNTHETIC DATA GENERATION PIPELINE (TabGAN + DAE + Original)

STEP 1: Loading models
--------------------------------------------------------------------------------
✓ Loaded TabGAN model: ../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_none.pkl
✓ Loaded TabGAN model: ../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_syn-flood.pkl
✓ Loaded TabGAN model: ../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_Backdoor.pkl
✓ Loaded DAE checkpoint: ../data/dataset_synthetic extension/hyperparameter_tuning/models/dae_none.pt
✓ Loaded DAE checkpoint: ../data/dataset_synthetic extension/hyperparameter_tuning/models/dae_syn-flood.pt
✓ Loaded DAE checkpoint: ../data/dataset_synthetic extension/hyperparameter_tuning/models/dae_Backdoor.pt

STEP 2: Loading original data & fitting Scalers
--------------------------------------------------------------------------------
✓ Loaded original data: synthetic_exte

Generating blocks:   0%|                                | 0/26 [00:00<?, ?it/s]

Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/100 [00:00<?, ?it/s]

Generating blocks:   4%|▉                       | 1/26 [00:07<03:01,  7.25s/it]

...

Generating blocks:  96%|██████████████████████ | 25/26 [04:14<00:10, 10.70s/it]

Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/100 [00:00<?, ?it/s]

Generating blocks: 100%|███████████████████████| 26/26 [04:29<00:00, 10.36s/it]



NEGATIVE VALUE CHECK REPORT
Feature 'shunt_voltage': 0 negative values
Feature 'bus_voltage_V': 0 negative values
Feature 'current_mA': 0 negative values
Feature 'power_mW': 0 negative values

Final dataset sample (Head):
   shunt_voltage  bus_voltage_V  current_mA     power_mW     State Attack  \
0     721.530762       5.184332  856.167969  3411.661621  charging   none   
1     667.000000       5.175420  848.000000  2948.000000  charging   none   
2     877.989319       5.193930  621.554382  2989.835205  charging   none   
3     592.000000       5.195589  554.000000  4981.000000  charging   none   
4     795.000000       5.197000  516.000000  2580.000000  charging   none   
5     590.906494       5.191048  707.063049  3464.315430  charging   none   
6     527.000000       5.188395  619.000000  3581.000000  charging   none   
7     581.243713       5.186493  648.777100  3479.008301  charging   none   
8     692.000000       5.188195  672.000000  3223.000000  charging   none   
9     5

In [20]:
import pickle
from pathlib import Path

# Load one TabGAN model to inspect
model_path = Path("../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_none.pkl")
with open(model_path, "rb") as f:
    model = pickle.load(f)

print(f"Model type: {type(model)}")
print(f"Model attributes/methods: {[attr for attr in dir(model) if not attr.startswith('_')]}")

# Check if it has common methods
print(f"\nHas 'sample': {hasattr(model, 'sample')}")
print(f"Has 'generate': {hasattr(model, 'generate')}")
print(f"Has 'predict': {hasattr(model, 'predict')}")
print(f"Has 'forward': {hasattr(model, 'forward')}")

# If it's a GAN, check for generator/discriminator
if hasattr(model, 'generator'):
    print(f"\nHas generator: {type(model.generator)}")
    print(f"Generator methods: {[attr for attr in dir(model.generator) if not attr.startswith('_')]}")

Model type: <class 'tabgan.sampler.GANGenerator'>
Model attributes/methods: ['args', 'generate_data_pipe', 'get_object_generator', 'kwargs', 'last_timing_']

Has 'sample': False
Has 'generate': False
Has 'predict': False
Has 'forward': False


In [21]:
import pickle
from pathlib import Path
import inspect

# Load one TabGAN model
model_path = Path("../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_none.pkl")
with open(model_path, "rb") as f:
    model = pickle.load(f)

# Check the signature of generate_data_pipe
print("generate_data_pipe signature:")
print(inspect.signature(model.generate_data_pipe))

# Check if get_object_generator gives us the actual generator
print("\nGenerator object:")
generator = model.get_object_generator()
print(f"Type: {type(generator)}")
print(f"Methods: {[attr for attr in dir(generator) if not attr.startswith('_')]}")

generate_data_pipe signature:
(train_df: pandas.DataFrame, target: pandas.DataFrame, test_df: pandas.DataFrame, deep_copy: bool = True, only_adversarial: bool = False, use_adversarial: bool = True, only_generated_data: bool = False, constraints: Optional[List] = None) -> Tuple[pandas.DataFrame, pandas.DataFrame]

Generator object:
Type: <class 'tabgan.sampler.SamplerGAN'>
Methods: ['TEMP_TARGET', 'adversarial_filtering', 'adversarial_model_params', 'bot_filter_quantile', 'cat_cols', 'check_params', 'conditional_columns', 'gen_params', 'gen_x_times', 'generate_data', 'get_generated_shape', 'handle_generated_data', 'is_post_process', 'llm_api_config', 'only_generated_data', 'postprocess_data', 'pregeneration_frac', 'preprocess_data', 'preprocess_data_df', 'text_generating_columns', 'top_filter_quantile']


In [22]:
import pickle
from pathlib import Path
import inspect

model_path = Path("../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_none.pkl")
with open(model_path, "rb") as f:
    model = pickle.load(f)

generator = model.get_object_generator()
print("generate_data signature:")
print(inspect.signature(generator.generate_data))
print(f"\ngen_x_times value: {generator.gen_x_times if hasattr(generator, 'gen_x_times') else 'N/A'}")

generate_data signature:
(train_df, target, test_df, only_generated_data: bool) -> Tuple[pandas.DataFrame, pandas.DataFrame]

gen_x_times value: 0.75


In [23]:
import torch
from pathlib import Path

# Load one DAE checkpoint
dae_path = Path("../data/dataset_synthetic extension/hyperparameter_tuning/models/dae_none.pt")
checkpoint = torch.load(dae_path, map_location="cpu")

print(f"Checkpoint type: {type(checkpoint)}")
if isinstance(checkpoint, dict):
    print(f"Keys: {checkpoint.keys()}")
    
    # Check if it's a state_dict
    if 'state_dict' in checkpoint:
        print(f"\nState dict keys: {list(checkpoint['state_dict'].keys())[:10]}")
    else:
        print(f"\nState dict keys (first 10): {list(checkpoint.keys())[:10]}")
    
    # Try to infer architecture from layer names
    if isinstance(checkpoint, dict):
        print("\nLayer shapes:")
        for key, value in (checkpoint.get('state_dict', checkpoint) if 'state_dict' in checkpoint else checkpoint).items():
            if isinstance(value, torch.Tensor):
                print(f"  {key}: {value.shape}")

Checkpoint type: <class 'dict'>
Keys: dict_keys(['epoch', 'model_state_dict', 'model_config'])

State dict keys (first 10): ['epoch', 'model_state_dict', 'model_config']

Layer shapes:


In [24]:
import torch
from pathlib import Path

dae_path = Path("../data/dataset_synthetic extension/hyperparameter_tuning/models/dae_none.pt")
checkpoint = torch.load(dae_path, map_location="cpu")

print("Model config:")
print(checkpoint.get('model_config', 'No config found'))

print("\nModel state dict keys (first 20):")
state_dict = checkpoint.get('model_state_dict', checkpoint)
for key in list(state_dict.keys())[:20]:
    if isinstance(state_dict[key], torch.Tensor):
        print(f"  {key}: {state_dict[key].shape}")
    else:
        print(f"  {key}: {type(state_dict[key])}")

Model config:
{'input_dim': 6, 'hidden_dims': (32, 16), 'latent_dim': 18, 'dropout': 0.05, 'alpha': 0.75, 'beta': 1.0, 'batch_size': 64, 'learning_rate': 0.002, 'weight_decay': 0.0, 'noise_factor': 0.03, 'denoise_val': True, 'denoise_test': False}

Model state dict keys (first 20):
  encoder.0.weight: torch.Size([32, 6])
  encoder.0.bias: torch.Size([32])
  encoder.3.weight: torch.Size([16, 32])
  encoder.3.bias: torch.Size([16])
  encoder.6.weight: torch.Size([18, 16])
  encoder.6.bias: torch.Size([18])
  decoder.0.weight: torch.Size([16, 18])
  decoder.0.bias: torch.Size([16])
  decoder.3.weight: torch.Size([32, 16])
  decoder.3.bias: torch.Size([32])
  decoder.6.weight: torch.Size([6, 32])
  decoder.6.bias: torch.Size([6])


In [25]:
import pickle
from pathlib import Path

model_path = Path("../data/dataset_synthetic extension/hyperparameter_tuning/models/tabgan_generator_none.pkl")
with open(model_path, "rb") as f:
    model = pickle.load(f)

generator = model.get_object_generator()

# Check available methods
print("Generator methods:")
for attr in dir(generator):
    if not attr.startswith('_') and callable(getattr(generator, attr)):
        print(f"  {attr}")

# Check if there's a way to sample without retraining
print(f"\nHas 'sample_data': {hasattr(generator, 'sample_data')}")
print(f"Has 'generate_new_data': {hasattr(generator, 'generate_new_data')}")
print(f"Has 'predict': {hasattr(generator, 'predict')}")

Generator methods:
  adversarial_filtering
  check_params
  generate_data
  get_generated_shape
  handle_generated_data
  postprocess_data
  preprocess_data
  preprocess_data_df

Has 'sample_data': False
Has 'generate_new_data': False
Has 'predict': False


In [26]:
# next steps:
- mache es dynamisch die die how many row generation
- orientiere dich an die blocks der csv und sortiere danach
- erstelle auch noch node B-H

SyntaxError: invalid syntax (2143679801.py, line 2)